# exp053 Stage 2A: BC2026 Finetune from Stage 1 Backbone (Colab Blackwell)

**Pipeline**: Stage 1 (XC pretrain) backbone → Stage 2A (BC2026 finetune) → 新 stream blend

## Input

- Stage 1 backbone: `/content/drive/MyDrive/kaggle/birdclef2026/exp052/ckpt/stage1_backbone_best.pth`
  - val_macro 0.9390 (Aves AUC 0.939 over 159 species)
- BC2026 train_audio (Kaggle Competition download)

## Design (Path A: Standalone)

| Item | Value | Reason |
|---|---|---|
| Backbone | Stage 1 ckpt load (XC pretrained EffNetV2-S) | 鳥音響 general 学習済 |
| Head | **Reinit** (random init) | BC2026 234 distribution に合わせ scratch から |
| Data | BC2026 train_audio (35k, 234 species) | 全 taxa 学習 |
| Loss | BCEWithLogitsLoss + label smooth | 多ラベル + soft target |
| Optimizer | AdamW、**differential LR** | backbone=5e-5、head=5e-4 |
| Scheduler | CosineAnnealingLR T_max=20 | 標準 |
| Epochs | **20** | 短い、backbone 既学習済 |
| Batch | 256 (Blackwell) | |
| Aug | Spec mixup + SpecAugment + wave mixup | Stage 1 同 |
| Label smooth | 0.05 | |

## 期待

- val_ns22: **0.92-0.94**
- standalone LB: **0.92-0.94** (Path A standalone target)
- 4-way blend (exp048 + Stage 2A) LB: **0.951-0.955** (gold border 接近)
- Blackwell 訓練時間: **30-45 min** (Stage 1 が想定の 1/10 で終了したので Stage 2A も短い)

## Output

- Drive: `/content/drive/MyDrive/kaggle/birdclef2026/exp053/ckpt/stage2a_best.pth`
- Kaggle Dataset: `maekeso/birdclef2026-exp053-stage2a-effv2s`


In [1]:
# ============================================================
# Cell 1: Install + Drive mount + Kaggle API setup
# ============================================================
!pip install -q timm kaggle soundfile librosa torchaudio tqdm

from google.colab import drive
drive.mount("/content/drive")

import os, json, shutil
from pathlib import Path

# Drive paths
DRIVE_ROOT = Path("/content/drive/MyDrive/kaggle/birdclef2026/exp053")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_DIR = DRIVE_ROOT / "ckpt"
CKPT_DIR.mkdir(exist_ok=True)

# Stage 1 backbone path
STAGE1_CKPT = Path("/content/drive/MyDrive/kaggle/birdclef2026/exp052/ckpt/stage1_backbone_best.pth")
assert STAGE1_CKPT.exists(), f"Stage 1 ckpt not found: {STAGE1_CKPT}"
print(f"Stage 1 ckpt found: {STAGE1_CKPT} ({STAGE1_CKPT.stat().st_size/1e6:.1f} MB)")

# Kaggle credentials - try multiple locations
KAGGLE_JSON_CANDIDATES = [
    Path("/content/drive/MyDrive/kaggle/birdclef2026/kaggle.json"),
    Path("/content/drive/MyDrive/kaggle.json"),
    Path("/content/drive/MyDrive/.kaggle/kaggle.json"),
    Path("/content/kaggle.json"),
    Path.home() / ".kaggle" / "kaggle.json",
]
KAGGLE_JSON_PATH = next((p for p in KAGGLE_JSON_CANDIDATES if p.exists()), None)
assert KAGGLE_JSON_PATH is not None, "kaggle.json not found"
print(f"Using kaggle.json from: {KAGGLE_JSON_PATH}")

# Set KAGGLE_API_TOKEN env var (CLAUDE.md memory)
creds = json.loads(KAGGLE_JSON_PATH.read_text())
os.environ["KAGGLE_API_TOKEN"] = creds["key"]
os.makedirs(Path.home() / ".kaggle", exist_ok=True)
shutil.copy(KAGGLE_JSON_PATH, Path.home() / ".kaggle" / "kaggle.json")
os.chmod(Path.home() / ".kaggle" / "kaggle.json", 0o600)
print(f"OK KAGGLE_API_TOKEN env var set")


Mounted at /content/drive
Stage 1 ckpt found: /content/drive/MyDrive/kaggle/birdclef2026/exp052/ckpt/stage1_backbone_best.pth (84.0 MB)
Using kaggle.json from: /content/drive/MyDrive/kaggle/birdclef2026/kaggle.json
OK KAGGLE_API_TOKEN env var set


In [ ]:
# ============================================================
# Cell 2: Download BC2026 competition data (full zip + selective unzip)
# ============================================================
import subprocess
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
print("OK Kaggle SDK authenticated")

BC_ROOT = Path("/content/bc2026")
BC_ROOT.mkdir(exist_ok=True)
TRAIN_AUDIO = BC_ROOT / "train_audio"

# Check if already downloaded
n_audio = len(list(TRAIN_AUDIO.rglob("*.ogg"))) if TRAIN_AUDIO.exists() else 0
if n_audio > 30000:
    print(f"train_audio already exists ({n_audio} .ogg files), skip download")
else:
    print(f"Downloading entire BC2026 competition (~30-50 GB, ~15-30 min)...")
    print(f"  Note: Kaggle API doesn't allow directory-level download")
    print(f"        Must download full zip + extract selectively")

    # Download full competition zip
    api.competition_download_files("birdclef-2026", path=str(BC_ROOT), quiet=False)

    # Find downloaded zip
    zip_files = list(BC_ROOT.glob("birdclef-2026.zip"))
    if not zip_files:
        zip_files = list(BC_ROOT.glob("*.zip"))
    if not zip_files:
        raise FileNotFoundError(f"No zip found in {BC_ROOT}")
    zip_path = zip_files[0]
    print(f"\n  Downloaded zip: {zip_path} ({zip_path.stat().st_size/1e9:.2f} GB)")

    # Selective unzip (train_audio + meta only, skip test_soundscapes 20+GB)
    import zipfile
    print(f"\nExtracting train_audio + meta (skip test_soundscapes to save disk)...")
    with zipfile.ZipFile(zip_path) as zf:
        all_names = zf.namelist()
        # Filter: train_audio/, meta CSVs
        needed = [n for n in all_names
                  if n.startswith("train_audio/")
                  or n in ("train.csv", "taxonomy.csv", "train_soundscapes_labels.csv",
                           "sample_submission.csv", "recording_location.txt")]
        print(f"  Total files in zip: {len(all_names)}")
        print(f"  Extracting: {len(needed)} (train_audio + meta)")
        for i, name in enumerate(needed):
            if i % 5000 == 0:
                print(f"    [{i}/{len(needed)}]")
            zf.extract(name, path=str(BC_ROOT))

    # Delete zip to free disk
    print(f"\n  Deleting zip to free disk...")
    zip_path.unlink()

# Verify
import pandas as pd
train_csv = BC_ROOT / "train.csv"
assert train_csv.exists(), f"train.csv missing: {train_csv}"
train_df = pd.read_csv(train_csv)
n_audio = len(list(TRAIN_AUDIO.rglob("*.ogg"))) if TRAIN_AUDIO.exists() else 0
print(f"\n=== Verify ===")
print(f"  train.csv: {len(train_df)} rows")
print(f"  train_audio: {n_audio} .ogg files")
print(f"  taxonomy.csv: exists={(BC_ROOT/'taxonomy.csv').exists()}")
print(f"  train_soundscapes_labels.csv: exists={(BC_ROOT/'train_soundscapes_labels.csv').exists()}")
assert n_audio > 30000, f"Expected >30k audio files, got {n_audio}"


OK Kaggle SDK authenticated
  Note: Kaggle API doesn't allow directory-level download
        Must download full zip + extract selectively


100%|██████████| 15.0G/15.0G [06:13<00:00, 43.0MB/s]




  Downloaded zip: /content/bc2026/birdclef-2026.zip (16.07 GB)

Extracting train_audio + meta (skip test_soundscapes to save disk)...
  Total files in zip: 46213
  Extracting: 35554 (train_audio + meta)
    [0/35554]
    [5000/35554]
    [10000/35554]
    [15000/35554]
    [20000/35554]
    [25000/35554]
    [30000/35554]
    [35000/35554]

  Deleting zip to free disk...

=== Verify ===
  train.csv: 35549 rows
  train_audio: 35549 .ogg files
  taxonomy.csv: exists=True
  train_soundscapes_labels.csv: exists=True


In [ ]:
# ============================================================
# Cell 3: Setup + GPU check
# ============================================================
import os, sys, json, time, math, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
import timm
import soundfile as sf
import librosa
from tqdm.auto import tqdm

print(f"torch: {torch.__version__}, cuda: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name} ({p.total_memory/1e9:.1f} GB)")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


torch: 2.10.0+cu128, cuda: True
  [0] NVIDIA RTX PRO 6000 Blackwell Server Edition (102.0 GB)


In [ ]:
# ============================================================
# Cell 4: BC2026 species list (234 species)
# ============================================================
__BC2026_SPECIES = [
    ('Guyalna cuta', '1161364', 'Insecta'),
    ('Caiman yacare', '116570', 'Reptilia'),
    ('Leptodactylus luctator', '1176823', 'Amphibia'),
    ('Adenomera guarani', '1491113', 'Amphibia'),
    ('Lysapsus limellum', '1595929', 'Amphibia'),
    ('Equus caballus', '209233', 'Mammalia'),
    ('Leptodactylus syphax', '22930', 'Amphibia'),
    ('Leptodactylus mystacinus', '22956', 'Amphibia'),
    ('Leptodactylus podicipinus', '22961', 'Amphibia'),
    ('Leptodactylus elenae', '22967', 'Amphibia'),
    ('Leptodactylus fuscus', '22973', 'Amphibia'),
    ('Leptodactylus labyrinthicus', '22983', 'Amphibia'),
    ('Leptodactylus petersii', '22985', 'Amphibia'),
    ('Physalaemus centralis', '23150', 'Amphibia'),
    ('Physalaemus albifrons', '23154', 'Amphibia'),
    ('Physalaemus albonotatus', '23158', 'Amphibia'),
    ('Pseudopaludicola mystacalis', '23176', 'Amphibia'),
    ('Phyllomedusa sauvagii', '23724', 'Amphibia'),
    ('Scinax nasicus', '24279', 'Amphibia'),
    ('Scinax fuscovarius', '24285', 'Amphibia'),
    ('Scinax fuscomarginatus', '24287', 'Amphibia'),
    ('Scinax acuminatus', '24321', 'Amphibia'),
    ('Quesada gigas', '244024', 'Insecta'),
    ('Chiasmocleis mehelyi', '25073', 'Amphibia'),
    ('Elachistocleis bicolor', '25092', 'Amphibia'),
    ('Dermatonotus muelleri', '25214', 'Amphibia'),
    ('Physalaemus biligonigerus', '326272', 'Amphibia'),
    ('Panthera onca', '41970', 'Mammalia'),
    ('Alouatta caraya', '43435', 'Mammalia'),
    ('Canis familiaris', '47144', 'Mammalia'),
    ('Insect son01', '47158son01', 'Insecta'),
    ('Insect son02', '47158son02', 'Insecta'),
    ('Insect son03', '47158son03', 'Insecta'),
    ('Insect son04', '47158son04', 'Insecta'),
    ('Insect son05', '47158son05', 'Insecta'),
    ('Insect son06', '47158son06', 'Insecta'),
    ('Insect son07', '47158son07', 'Insecta'),
    ('Insect son08', '47158son08', 'Insecta'),
    ('Insect son09', '47158son09', 'Insecta'),
    ('Insect son10', '47158son10', 'Insecta'),
    ('Insect son11', '47158son11', 'Insecta'),
    ('Insect son12', '47158son12', 'Insecta'),
    ('Insect son13', '47158son13', 'Insecta'),
    ('Insect son14', '47158son14', 'Insecta'),
    ('Insect son15', '47158son15', 'Insecta'),
    ('Insect son16', '47158son16', 'Insecta'),
    ('Insect son17', '47158son17', 'Insecta'),
    ('Insect son18', '47158son18', 'Insecta'),
    ('Insect son19', '47158son19', 'Insecta'),
    ('Insect son20', '47158son20', 'Insecta'),
    ('Insect son21', '47158son21', 'Insecta'),
    ('Insect son22', '47158son22', 'Insecta'),
    ('Insect son23', '47158son23', 'Insecta'),
    ('Insect son24', '47158son24', 'Insecta'),
    ('Insect son25', '47158son25', 'Insecta'),
    ('Physalaemus nattereri', '476521', 'Amphibia'),
    ('Sapajus cay', '516975', 'Mammalia'),
    ('Pithecopus azureus', '517063', 'Amphibia'),
    ('Boana lundii', '555123', 'Amphibia'),
    ('Boana punctata', '555145', 'Amphibia'),
    ('Boana raniceps', '555146', 'Amphibia'),
    ('Ameerega picta', '64898', 'Amphibia'),
    ('Dendropsophus minutus', '65377', 'Amphibia'),
    ('Dendropsophus nanus', '65380', 'Amphibia'),
    ('Pseudis platensis', '66971', 'Amphibia'),
    ('Rhinella diptycha', '67107', 'Amphibia'),
    ('Trachycephalus typhonius', '67252', 'Amphibia'),
    ('Leptodactylus macrosternum', '70711', 'Amphibia'),
    ('Plecturocebus pallescens', '738183', 'Mammalia'),
    ('Bos taurus', '74113', 'Mammalia'),
    ('Mico melanurus', '74580', 'Mammalia'),
    ('Prionacris erosa', '760266', 'Insecta'),
    ('Hylophilus pectoralis', 'ashgre1', 'Aves'),
    ('Mustelirallus albicollis', 'astcra1', 'Aves'),
    ('Crax fasciolata', 'bafcur1', 'Aves'),
    ('Micrastur ruficollis', 'baffal1', 'Aves'),
    ('Coereba flaveola', 'banana', 'Aves'),
    ('Thamnophilus doliatus', 'barant1', 'Aves'),
    ('Procnias nudicollis', 'batbel1', 'Aves'),
    ('Ara ararauna', 'baymac', 'Aves'),
    ('Dendrocygna autumnalis', 'bbwduc', 'Aves'),
    ('Microspingus melanoleucus', 'bcwfin2', 'Aves'),
    ('Donacobius atricapilla', 'bkcdon', 'Aves'),
    ('Aratinga nenday', 'bkhpar', 'Aves'),
    ('Busarellus nigricollis', 'blchaw1', 'Aves'),
    ('Spizaetus tyrannus', 'blheag1', 'Aves'),
    ('Tityra cayana', 'blttit1', 'Aves'),
    ('Myiarchus tyrannulus', 'bncfly', 'Aves'),
    ('Megarynchus pitangua', 'bobfly1', 'Aves'),
    ('Progne tapera', 'brcmar1', 'Aves'),
    ('Tyto furcata', 'brnowl', 'Aves'),
    ('Momotus momota', 'bucmot4', 'Aves'),
    ('Thectocercus acuticaudatus', 'bucpar', 'Aves'),
    ('Amazona aestiva', 'bufpar', 'Aves'),
    ('Theristicus caudatus', 'bunibi1', 'Aves'),
    ('Athene cunicularia', 'burowl', 'Aves'),
    ('Colaptes campestris', 'camfli1', 'Aves'),
    ('Ortalis canicollis', 'chacha1', 'Aves'),
    ('Mimus saturninus', 'chbmoc1', 'Aves'),
    ('Gnorimopsar chopi', 'chobla1', 'Aves'),
    ('Conirostrum speciosum', 'chvcon1', 'Aves'),
    ('Synallaxis hypospodia', 'cibspi1', 'Aves'),
    ('Micrastur semitorquatus', 'coffal1', 'Aves'),
    ('Nyctidromus albicollis', 'compau', 'Aves'),
    ('Nyctibius griseus', 'compot1', 'Aves'),
    ('Turdus amaurochalinus', 'crbthr1', 'Aves'),
    ('Pachyramphus validus', 'crebec1', 'Aves'),
    ('Taoniscus nanus', 'dwatin1', 'Aves'),
    ('Icterus pyrrhopterus', 'epaori4', 'Aves'),
    ('Lathrotriccus euleri', 'eulfly1', 'Aves'),
    ('Cantorchilus guarayanus', 'fabwre1', 'Aves'),
    ('Glaucidium brasilianum', 'fepowl', 'Aves'),
    ('Machaeropterus pyrocephalus', 'ficman1', 'Aves'),
    ('Myiothlypis flaveola', 'flawar1', 'Aves'),
    ('Tyrannus savana', 'fotfly', 'Aves'),
    ('Cnemotriccus fuscatus', 'fusfly1', 'Aves'),
    ('Hylocharis chrysura', 'gilhum1', 'Aves'),
    ('Aramides ypecaha', 'giwrai1', 'Aves'),
    ('Chionomesa fimbriata', 'glteme1', 'Aves'),
    ('Saltator coerulescens', 'grasal3', 'Aves'),
    ('Crotophaga major', 'greani1', 'Aves'),
    ('Taraba major', 'greant1', 'Aves'),
    ('Myiopagis viridicata', 'greela', 'Aves'),
    ('Pitangus sulphuratus', 'grekis', 'Aves'),
    ('Nyctibius grandis', 'grepot1', 'Aves'),
    ('Phacellodomus ruber', 'gretho2', 'Aves'),
    ('Tringa melanoleuca', 'greyel', 'Aves'),
    ('Leptotila rufaxilla', 'grfdov1', 'Aves'),
    ('Eucometis penicillata', 'grhtan1', 'Aves'),
    ('Aramides cajaneus', 'gycwor1', 'Aves'),
    ('Anhima cornuta', 'horscr1', 'Aves'),
    ('Passer domesticus', 'houspa', 'Aves'),
    ('Anodorhynchus hyacinthinus', 'hyamac1', 'Aves'),
    ('Elaenia spectabilis', 'larela1', 'Aves'),
    ('Elaenia chiriquensis', 'lesela1', 'Aves'),
    ('Emberizoides ypiranganus', 'lesgrf1', 'Aves'),
    ('Aramus guarauna', 'limpki', 'Aves'),
    ('Dryocopus lineatus', 'linwoo1', 'Aves'),
    ('Coccycua minuta', 'litcuc2', 'Aves'),
    ('Setopagis parvula', 'litnig1', 'Aves'),
    ('Pyrrhura frontalis', 'mabpar', 'Aves'),
    ('Cercomacra melanaria', 'magant1', 'Aves'),
    ('Cissopis leverianus', 'magtan2', 'Aves'),
    ('Polioptila dumicola', 'masgna1', 'Aves'),
    ('Chordeiles nacunda', 'nacnig1', 'Aves'),
    ('Rufirallus schomburgkii', 'ocecra1', 'Aves'),
    ('Sittasomus griseicapillus', 'oliwoo1', 'Aves'),
    ('Icterus croconotus', 'orbtro3', 'Aves'),
    ('Amazona amazonica', 'orwpar', 'Aves'),
    ('Pandion haliaetus', 'osprey', 'Aves'),
    ('Synallaxis albescens', 'pabspi1', 'Aves'),
    ('Furnarius leucopus', 'palhor3', 'Aves'),
    ('Thraupis palmarum', 'paltan1', 'Aves'),
    ('Dromococcyx phasianellus', 'phecuc1', 'Aves'),
    ('Patagioenas picazuro', 'picpig2', 'Aves'),
    ('Legatus leucophaius', 'pirfly1', 'Aves'),
    ('Thamnophilus pelzelni', 'plasla1', 'Aves'),
    ('Inezia inornata', 'platyr1', 'Aves'),
    ('Cyanocorax chrysops', 'plcjay1', 'Aves'),
    ('Theristicus caerulescens', 'pluibi1', 'Aves'),
    ('Cyanocorax cyanomelas', 'purjay1', 'Aves'),
    ('Hemitriccus margaritaceiventer', 'pvttyr1', 'Aves'),
    ('Ara chloropterus', 'ragmac1', 'Aves'),
    ('Campylorhamphus trochilirostris', 'rebscy1', 'Aves'),
    ('Coryphospingus cucullatus', 'recfin1', 'Aves'),
    ('Gallus gallus', 'redjun', 'Aves'),
    ('Cariama cristata', 'relser1', 'Aves'),
    ('Megaceryle torquata', 'rinkin1', 'Aves'),
    ('Myiothlypis rivularis', 'rivwar1', 'Aves'),
    ('Rupornis magnirostris', 'roahaw', 'Aves'),
    ('Turdus rufiventris', 'rubthr1', 'Aves'),
    ('Pseudoseisura unirufa', 'rufcac2', 'Aves'),
    ('Casiornis rufus', 'rufcas2', 'Aves'),
    ('Conopophaga lineata', 'rufgna3', 'Aves'),
    ('Furnarius rufus', 'rufhor2', 'Aves'),
    ('Antrostomus rufus', 'rufnig1', 'Aves'),
    ('Phacellodomus rufifrons', 'ruftho1', 'Aves'),
    ('Poecilotriccus latirostris', 'ruftof1', 'Aves'),
    ('Myiozetetes cayanensis', 'rumfly1', 'Aves'),
    ('Tigrisoma lineatum', 'ruther1', 'Aves'),
    ('Galbula ruficauda', 'rutjac1', 'Aves'),
    ('Arremon flavirostris', 'sabspa1', 'Aves'),
    ('Sicalis flaveola', 'saffin', 'Aves'),
    ('Thraupis sayaca', 'saytan1', 'Aves'),
    ('Columbina squammata', 'scadov1', 'Aves'),
    ('Pionus maximiliani', 'schpar1', 'Aves'),
    ('Phaethornis eurynome', 'scther1', 'Aves'),
    ('Myiarchus ferox', 'shcfly1', 'Aves'),
    ('Accipiter striatus', 'shshaw', 'Aves'),
    ('Lurocalis semitorquatus', 'shtnig1', 'Aves'),
    ('Ramphocelus carbo', 'sibtan2', 'Aves'),
    ('Crotophaga ani', 'smbani', 'Aves'),
    ('Crypturellus parvirostris', 'smbtin1', 'Aves'),
    ('Cacicus solitarius', 'sobcac1', 'Aves'),
    ('Camptostoma obsoletum', 'sobtyr1', 'Aves'),
    ('Myiozetetes similis', 'socfly1', 'Aves'),
    ('Synallaxis frontalis', 'sofspi1', 'Aves'),
    ('Corythopis delalandi', 'souant1', 'Aves'),
    ('Vanellus chilensis', 'soulap1', 'Aves'),
    ('Chauna torquata', 'souscr1', 'Aves'),
    ('Hypoedaleus guttatus', 'spbant3', 'Aves'),
    ('Synallaxis spixi', 'spispi1', 'Aves'),
    ('Antiurus maculicaudus', 'sptnig1', 'Aves'),
    ('Piaya cayana', 'squcuc1', 'Aves'),
    ('Dendroplex picus', 'stbwoo2', 'Aves'),
    ('Tapera naevia', 'strcuc1', 'Aves'),
    ('Butorides striata', 'strher2', 'Aves'),
    ('Asio clamator', 'strowl1', 'Aves'),
    ('Eupetomena macroura', 'swthum1', 'Aves'),
    ('Chiroxiphia caudata', 'swtman1', 'Aves'),
    ('Crypturellus tataupa', 'tattin1', 'Aves'),
    ('Campylorhynchus turdinus', 'thlwre1', 'Aves'),
    ('Ramphastos toco', 'toctou1', 'Aves'),
    ('Tyrannus melancholicus', 'trokin', 'Aves'),
    ('Megascops choliba', 'trsowl', 'Aves'),
    ('Crypturellus undulatus', 'undtin1', 'Aves'),
    ('Thamnophilus caerulescens', 'varant1', 'Aves'),
    ('Jacana jacana', 'watjac1', 'Aves'),
    ('Pyriglena maura', 'wesfie1', 'Aves'),
    ('Dendrocygna viduata', 'wfwduc1', 'Aves'),
    ('Biatas nigropectus', 'whbant2', 'Aves'),
    ('Myiothlypis leucoblephara', 'whbwar2', 'Aves'),
    ('Melanerpes candidus', 'whiwoo1', 'Aves'),
    ('Synallaxis albilora', 'whlspi1', 'Aves'),
    ('Cyanocorax cyanopogon', 'whnjay1', 'Aves'),
    ('Leptotila verreauxi', 'whtdov', 'Aves'),
    ('Picumnus albosquamatus', 'whwpic1', 'Aves'),
    ('Caracara plancus', 'y00678', 'Aves'),
    ('Paroaria capitata', 'yebcar', 'Aves'),
    ('Elaenia flavogaster', 'yebela1', 'Aves'),
    ('Primolius auricollis', 'yecmac', 'Aves'),
    ('Brotogeris chiriri', 'yecpar', 'Aves'),
    ('Daptrius chimachima', 'yehcar1', 'Aves'),
    ('Tolmomyias sulphurescens', 'yeofly1', 'Aves'),
]
species_df = pd.DataFrame(__BC2026_SPECIES, columns=["scientific_name", "primary_label", "class_name"])
PRIMARY_LABELS = species_df["primary_label"].tolist()
LABEL_TO_IDX = {l: i for i, l in enumerate(PRIMARY_LABELS)}
LABEL_TO_CLASS = dict(zip(species_df['primary_label'], species_df['class_name']))
N_CLASSES = len(PRIMARY_LABELS)
print(f"BC2026 species: {N_CLASSES}")
print(f"  taxa: {species_df['class_name'].value_counts().to_dict()}")


BC2026 species: 234
  taxa: {'Aves': 162, 'Amphibia': 35, 'Insecta': 28, 'Mammalia': 8, 'Reptilia': 1}


In [ ]:
# ============================================================
# Cell 5: Config
# ============================================================
class CFG:
    SEED = 42
    BACKBONE = "tf_efficientnetv2_s.in21k_ft_in1k"
    N_CLASSES = N_CLASSES

    SR = 32000
    CHUNK_SEC = 5
    CHUNK_LEN = SR * CHUNK_SEC

    N_MELS = 128
    N_FFT = 2048
    HOP = 512
    FMIN = 20
    FMAX = 16000

    # Train
    EPOCHS = 20
    BATCH_SIZE = 256
    LR_BACKBONE = 5e-5   # ★ small (Stage 1 既学習済を保持)
    LR_HEAD = 5e-4       # ★ large (head は scratch)
    WD = 1e-4
    NUM_WORKERS = 4

    MIXUP_PROB = 0.5
    MIXUP_ALPHA = 1.0
    WAVE_MIXUP_PROB = 0.3
    FREQ_MASK = 30
    TIME_MASK = 40
    LABEL_SMOOTH = 0.05

    BC_ROOT = Path("/content/bc2026")
    TRAIN_AUDIO = BC_ROOT / "train_audio"
    TRAIN_CSV = BC_ROOT / "train.csv"

    CKPT_DIR = CKPT_DIR
    BEST_CKPT = CKPT_DIR / "stage2a_best.pth"
    LAST_CKPT = CKPT_DIR / "stage2a_last.pth"
    HIST_JSON = CKPT_DIR / "stage2a_history.json"

torch.manual_seed(CFG.SEED)
np.random.seed(CFG.SEED)
random.seed(CFG.SEED)
torch.backends.cudnn.benchmark = True
print(f"Backbone: {CFG.BACKBONE}")
print(f"Train: {CFG.EPOCHS} ep × batch {CFG.BATCH_SIZE}")
print(f"LR: backbone={CFG.LR_BACKBONE} head={CFG.LR_HEAD}")
print(f"Output: {CFG.CKPT_DIR}")


Backbone: tf_efficientnetv2_s.in21k_ft_in1k
Train: 20 ep × batch 256
LR: backbone=5e-05 head=0.0005
Output: /content/drive/MyDrive/kaggle/birdclef2026/exp053/ckpt


In [ ]:
# ============================================================
# Cell 6: Build BC2026 train DataFrame
# ============================================================
train_df = pd.read_csv(CFG.TRAIN_CSV)
print(f"BC2026 train: {len(train_df)} rows")
print(f"  columns: {train_df.columns.tolist()[:10]}")

# Build audio path
def _resolve_audio_path(row):
    fname = row["filename"]
    if not str(fname).endswith(".ogg"):
        fname = fname + ".ogg"
    return CFG.TRAIN_AUDIO / fname
train_df["audio_path"] = train_df.apply(_resolve_audio_path, axis=1)
n_exists = train_df["audio_path"].apply(lambda p: p.exists()).sum()
print(f"  audio exists: {n_exists}/{len(train_df)}")

# Sample distribution
print(f"  taxa breakdown:")
print(train_df['class_name'].value_counts())


BC2026 train: 35549 rows
  columns: ['primary_label', 'secondary_labels', 'type', 'latitude', 'longitude', 'scientific_name', 'common_name', 'class_name', 'inat_taxon_id', 'author']
  audio exists: 35549/35549
  taxa breakdown:
class_name
Aves        34799
Amphibia      451
Insecta       199
Mammalia       99
Reptilia        1
Name: count, dtype: int64


In [ ]:
# ============================================================
# Cell 7: BC2026 Dataset (with secondary labels)
# ============================================================
class BC2026Dataset(Dataset):
    def __init__(self, df, sr=CFG.SR, chunk_len=CFG.CHUNK_LEN, training=True,
                 label_smooth=CFG.LABEL_SMOOTH):
        self.df = df.reset_index(drop=True)
        self.sr = sr
        self.chunk_len = chunk_len
        self.training = training
        self.label_smooth = label_smooth

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = row["audio_path"]
        try:
            with sf.SoundFile(str(audio_path)) as f:
                sr = f.samplerate
                src_chunk = int(self.chunk_len * sr / self.sr)
                total = f.frames
                if total > src_chunk:
                    if self.training:
                        start = np.random.randint(0, total - src_chunk + 1)
                    else:
                        start = (total - src_chunk) // 2
                    f.seek(start)
                    wav = f.read(src_chunk, dtype="float32")
                else:
                    wav = f.read(dtype="float32")
            if wav.ndim > 1:
                wav = wav.mean(axis=1)
            if sr != self.sr:
                wav = librosa.resample(wav, orig_sr=sr, target_sr=self.sr)
        except Exception:
            wav = np.zeros(self.chunk_len, dtype=np.float32)

        if len(wav) >= self.chunk_len:
            wav = wav[:self.chunk_len]
        else:
            wav = np.pad(wav, (0, self.chunk_len - len(wav)))

        # Multi-label with label smoothing (primary + secondary)
        label = np.full(N_CLASSES, self.label_smooth / 2, dtype=np.float32)
        primary = row["primary_label"]
        if primary in LABEL_TO_IDX:
            label[LABEL_TO_IDX[primary]] = 1.0 - self.label_smooth / 2

        # secondary_labels (BC2026 has this column)
        sec = row.get("secondary_labels", "[]")
        if isinstance(sec, str) and sec not in ("[]", "", "nan"):
            try:
                sec_list = eval(sec) if sec.startswith("[") else []
                for s in sec_list:
                    if s in LABEL_TO_IDX:
                        label[LABEL_TO_IDX[s]] = 1.0 - self.label_smooth / 2
            except Exception:
                pass

        return torch.from_numpy(wav), torch.from_numpy(label)


ds_check = BC2026Dataset(train_df.head(10), training=True)
wav, label = ds_check[0]
print(f"wav: {wav.shape}, range=[{wav.min().item():.3f}, {wav.max().item():.3f}]")
print(f"label: sum={label.sum().item():.2f}")


wav: torch.Size([160000]), range=[-0.216, 0.179]
label: sum=6.80


In [ ]:
# ============================================================
# Cell 8: Model + LOAD Stage 1 backbone
# ============================================================
import torchaudio

class MelExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP,
            n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX,
        )
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80.0)

    def forward(self, wav):
        mel = self.mel(wav)
        mel = self.db(mel)
        mel = torch.clamp(mel, -80.0, 0.0)
        mel = (mel + 40.0) / 40.0
        return mel


class SEDHead(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.att = nn.Linear(in_dim, n_classes)
        self.cla = nn.Linear(in_dim, n_classes)

    def forward(self, x):
        att = torch.tanh(self.att(x))
        cla = self.cla(x)
        norm_att = F.softmax(att, dim=1)
        clipwise = (norm_att * cla).sum(dim=1)
        return clipwise, cla


class SEDModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            CFG.BACKBONE, pretrained=False, in_chans=3,  # ★ no pretrained (Stage 1 で上書き)
            num_classes=0, global_pool="",
        )
        feat_dim = self.backbone.num_features
        self.head = SEDHead(feat_dim, CFG.N_CLASSES)

    def forward(self, mel):
        x = mel.unsqueeze(1).repeat(1, 3, 1, 1)
        feat = self.backbone(x)
        feat = feat.mean(dim=2)
        feat = feat.transpose(1, 2)
        clipwise, framewise = self.head(feat)
        return clipwise, framewise


# Build model + load Stage 1 backbone
model = SEDModel().to(DEVICE)
print(f"Model created, params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

# ★ Load Stage 1 backbone
stage1_ckpt = torch.load(STAGE1_CKPT, map_location=DEVICE, weights_only=False)
backbone_state = stage1_ckpt.get("backbone", stage1_ckpt.get("state_dict"))
if backbone_state is None:
    raise ValueError(f"Stage 1 ckpt missing backbone key, has: {list(stage1_ckpt.keys())}")
missing, unexpected = model.backbone.load_state_dict(backbone_state, strict=False)
print(f"\nStage 1 backbone loaded:")
print(f"  missing keys: {len(missing)} (head 等は無視 OK)")
print(f"  unexpected keys: {len(unexpected)}")
print(f"  Stage 1 val_ns22: {stage1_ckpt.get('val_ns22', 'n/a')}")
print(f"  Stage 1 epoch: {stage1_ckpt.get('epoch', 'n/a')}")
print(f"\n=> Head は scratch から学習開始 (random init)")

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
mel_extractor = MelExtractor().to(DEVICE)


Model created, params: 20.8M

Stage 1 backbone loaded:
  missing keys: 0 (head 等は無視 OK)
  unexpected keys: 0
  Stage 1 val_ns22: 0.9423327938232607
  Stage 1 epoch: 27

=> Head は scratch から学習開始 (random init)


In [ ]:
# ============================================================
# Cell 9: Augmentations (same as Stage 1)
# ============================================================
def spec_mixup(mel, label, alpha=CFG.MIXUP_ALPHA, p=CFG.MIXUP_PROB):
    if np.random.random() > p:
        return mel, label
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(mel.size(0), device=mel.device)
    mel_mix = lam * mel + (1 - lam) * mel[idx]
    label_mix = torch.maximum(label, label[idx])
    return mel_mix, label_mix

def wave_mixup(wav, label, alpha=0.5, p=CFG.WAVE_MIXUP_PROB):
    if np.random.random() > p:
        return wav, label
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(wav.size(0), device=wav.device)
    wav_mix = lam * wav + (1 - lam) * wav[idx]
    label_mix = torch.maximum(label, label[idx])
    return wav_mix, label_mix

def spec_augment(mel, freq_mask=CFG.FREQ_MASK, time_mask=CFG.TIME_MASK):
    B, F_, T = mel.shape
    for b in range(B):
        if freq_mask > 0:
            f = np.random.randint(0, freq_mask)
            f0 = np.random.randint(0, max(1, F_ - f))
            mel[b, f0:f0+f, :] = 0
        if time_mask > 0:
            t = np.random.randint(0, time_mask)
            t0 = np.random.randint(0, max(1, T - t))
            mel[b, :, t0:t0+t] = 0
    return mel


In [ ]:
# ============================================================
# Cell 10: Train/val split + dataloaders
# ============================================================
train_df_s = train_df[train_df["audio_path"].apply(lambda p: p.exists())].reset_index(drop=True)
print(f"Valid audio: {len(train_df_s)}/{len(train_df)}")

train_df_s = train_df_s.sample(frac=1.0, random_state=CFG.SEED).reset_index(drop=True)
n_val = int(len(train_df_s) * 0.05)
train_split = train_df_s.iloc[n_val:].reset_index(drop=True)
val_split = train_df_s.iloc[:n_val].reset_index(drop=True)
print(f"train: {len(train_split)}, val: {len(val_split)}")

train_ds = BC2026Dataset(train_split, training=True)
val_ds = BC2026Dataset(val_split, training=False)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True,
                          drop_last=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                         num_workers=CFG.NUM_WORKERS, pin_memory=True,
                         persistent_workers=True)
print(f"steps/ep: train={len(train_loader)}, val={len(val_loader)}")


Valid audio: 35549/35549
train: 33772, val: 1777
steps/ep: train=131, val=7


In [ ]:
# ============================================================
# Cell 11: Optimizer with differential LR (backbone slow, head fast)
# ============================================================
# Get backbone vs head params
if isinstance(model, nn.DataParallel):
    bk_params = model.module.backbone.parameters()
    hd_params = model.module.head.parameters()
else:
    bk_params = model.backbone.parameters()
    hd_params = model.head.parameters()

optimizer = torch.optim.AdamW([
    {"params": bk_params, "lr": CFG.LR_BACKBONE, "name": "backbone"},
    {"params": hd_params, "lr": CFG.LR_HEAD, "name": "head"},
], weight_decay=CFG.WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS)
loss_fn = nn.BCEWithLogitsLoss()
print(f"Optimizer: AdamW, LR backbone={CFG.LR_BACKBONE} head={CFG.LR_HEAD}")


Optimizer: AdamW, LR backbone=5e-05 head=0.0005


In [ ]:
# ============================================================
# Cell 12: rich_evaluate
# ============================================================
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def rich_evaluate(model, mel_ex, loader):
    model.eval()
    all_logits, all_labels = [], []
    for wav, label in tqdm(loader, desc="val", leave=False):
        wav = wav.to(DEVICE, non_blocking=True)
        mel = mel_ex(wav)
        with autocast("cuda", dtype=torch.bfloat16):
            logit, _ = model(mel)
        all_logits.append(logit.float().cpu())
        all_labels.append(label)
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    bin_labels = (labels > 0.5).astype(np.float32)

    per_class = []
    for c in range(bin_labels.shape[1]):
        n_pos = int(bin_labels[:, c].sum())
        if n_pos >= 2 and n_pos < bin_labels.shape[0]:
            try:
                auc = roc_auc_score(bin_labels[:, c], logits[:, c])
                per_class.append((PRIMARY_LABELS[c], float(auc), n_pos))
            except Exception:
                pass

    aucs_n22 = [auc for _, auc, n in per_class if n >= 22]
    val_ns22 = float(np.mean(aucs_n22)) if aucs_n22 else 0.0
    val_macro = float(np.mean([auc for _, auc, _ in per_class])) if per_class else 0.0

    taxon_aucs = {}
    for taxon in ["Aves", "Amphibia", "Insecta", "Mammalia", "Reptilia"]:
        aucs_in = [auc for label, auc, _ in per_class if LABEL_TO_CLASS.get(label) == taxon]
        taxon_aucs[taxon] = float(np.mean(aucs_in)) if aucs_in else float("nan")

    if aucs_n22:
        arr = np.array(aucs_n22)
        cs = {"n_valid": len(arr), "median": float(np.median(arr)),
              "p25": float(np.percentile(arr, 25)), "p75": float(np.percentile(arr, 75)),
              "n_above_05": int(np.sum(arr > 0.5)), "n_above_07": int(np.sum(arr > 0.7)),
              "n_above_09": int(np.sum(arr > 0.9)), "n_perfect": int(np.sum(arr >= 0.9999))}
    else:
        cs = {"n_valid": 0, "median": 0.0, "p25": 0.0, "p75": 0.0,
              "n_above_05": 0, "n_above_07": 0, "n_above_09": 0, "n_perfect": 0}
    return val_ns22, val_macro, taxon_aucs, cs


In [ ]:
# ============================================================
# Cell 13: Training loop (20 ep BC2026 finetune)
# ============================================================
import sys
def _p(msg):
    print(msg, flush=True)
    sys.stdout.flush()

# Smoke test
_p(f"[smoke] iter + forward + backward (bf16)...")
smoke_iter = iter(train_loader)
test_wav, test_label = next(smoke_iter)
test_wav_g = test_wav.to(DEVICE)
test_label_g = test_label.to(DEVICE)
test_mel = mel_extractor(test_wav_g)
_p(f"[smoke] mel: {test_mel.shape}")
with autocast("cuda", dtype=torch.bfloat16):
    test_logit, _ = model(test_mel)
    test_loss = loss_fn(test_logit, test_label_g)
_p(f"[smoke] loss={test_loss.item():.4f}")
test_loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
optimizer.zero_grad()
_p(f"[smoke] OK\n")
del smoke_iter, test_wav, test_label, test_wav_g, test_label_g, test_mel, test_logit, test_loss
torch.cuda.empty_cache()

history = {"train_loss": [], "val_ns22": [], "val_macro": [],
           "taxon_aucs": [], "class_stats": [], "lr_backbone": [], "lr_head": [], "elapsed_min": []}
best_val = 0.0
start_t = time.time()
nan_skip_count = 0

for epoch in range(1, CFG.EPOCHS + 1):
    ep_start = time.time()
    model.train()
    train_losses = []
    pbar = tqdm(train_loader, desc=f"Ep {epoch}/{CFG.EPOCHS}", leave=False, file=sys.stdout)
    for batch_idx, (wav, label) in enumerate(pbar):
        wav = wav.to(DEVICE, non_blocking=True)
        label = label.to(DEVICE, non_blocking=True)
        wav, label = wave_mixup(wav, label)
        mel = mel_extractor(wav)
        mel, label = spec_mixup(mel, label)
        mel = spec_augment(mel)

        with autocast("cuda", dtype=torch.bfloat16):
            logit, _ = model(mel)
            loss = loss_fn(logit, label)

        if torch.isnan(loss) or torch.isinf(loss):
            nan_skip_count += 1
            optimizer.zero_grad(set_to_none=True)
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        train_losses.append(loss.item())

        if batch_idx % 50 == 0:
            _p(f"[ep{epoch}] step {batch_idx}/{len(train_loader)} loss={np.mean(train_losses[-50:]):.4f}")
        del mel, logit, loss

    scheduler.step()
    torch.cuda.empty_cache()
    train_loss = float(np.mean(train_losses))

    val_ns22, val_macro, taxon_aucs, class_stats = rich_evaluate(model, mel_extractor, val_loader)
    lr_bk = optimizer.param_groups[0]["lr"]
    lr_hd = optimizer.param_groups[1]["lr"]
    elapsed_min = (time.time() - start_t) / 60
    ep_min = (time.time() - ep_start) / 60

    history["train_loss"].append(train_loss)
    history["val_ns22"].append(val_ns22)
    history["val_macro"].append(val_macro)
    history["taxon_aucs"].append(taxon_aucs)
    history["class_stats"].append(class_stats)
    history["lr_backbone"].append(lr_bk)
    history["lr_head"].append(lr_hd)
    history["elapsed_min"].append(elapsed_min)

    is_best = val_ns22 > best_val
    if is_best:
        best_val = val_ns22
        torch.save({
            "epoch": epoch, "val_ns22": val_ns22, "val_macro": val_macro,
            "taxon_aucs": taxon_aucs, "class_stats": class_stats,
            "state_dict": model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(),
            "config": {k: str(v) for k, v in CFG.__dict__.items() if not k.startswith("_")},
            "stage1_val_ns22": stage1_ckpt.get("val_ns22", "n/a"),
        }, CFG.BEST_CKPT)

    json.dump(history, open(CFG.HIST_JSON, "w"), indent=2)
    taxon_str = " ".join(f"{t}={v:.3f}" for t, v in taxon_aucs.items())
    cs = class_stats
    _p(f"=== Ep {epoch}/{CFG.EPOCHS}: loss={train_loss:.4f} val_ns22={val_ns22:.4f} val_macro={val_macro:.4f}"
       f" {'BEST' if is_best else ''} lr_bk={lr_bk:.2e} lr_hd={lr_hd:.2e} ({ep_min:.1f}min, total {elapsed_min:.1f}min) ===")
    _p(f"    taxon: {taxon_str}")
    _p(f"    class: n={cs['n_valid']} median={cs['median']:.3f} p25={cs['p25']:.3f} p75={cs['p75']:.3f}"
       f" #>0.5={cs['n_above_05']} #>0.7={cs['n_above_07']} #>0.9={cs['n_above_09']} #perfect={cs['n_perfect']}")

torch.save({
    "epoch": CFG.EPOCHS, "val_ns22": history["val_ns22"][-1],
    "state_dict": model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(),
}, CFG.LAST_CKPT)
_p(f"\n=== Stage 2A DONE. Best val_ns22={best_val:.4f} ===")


[smoke] iter + forward + backward (bf16)...
[smoke] mel: torch.Size([256, 128, 313])
[smoke] loss=0.7344
[smoke] OK



Ep 1/20:   0%|          | 0/131 [00:00<?, ?it/s]

[ep1] step 0/131 loss=0.5830
[ep1] step 50/131 loss=0.1779
[ep1] step 100/131 loss=0.1447


val:   0%|          | 0/7 [00:00<?, ?it/s]

=== Ep 1/20: loss=0.1605 val_ns22=0.8675 val_macro=0.8277 BEST lr_bk=4.97e-05 lr_hd=4.97e-04 (0.5min, total 0.5min) ===
    taxon: Aves=0.832 Amphibia=0.732 Insecta=0.922 Mammalia=nan Reptilia=nan
    class: n=35 median=0.871 p25=0.822 p75=0.934 #>0.5=35 #>0.7=34 #>0.9=12 #perfect=0


...

=== Ep 19/20: loss=0.1408 val_ns22=0.9087 val_macro=0.9188  lr_bk=3.08e-07 lr_hd=3.08e-06 (0.5min, total 10.3min) ===
    taxon: Aves=0.916 Amphibia=0.958 Insecta=0.999 Mammalia=nan Reptilia=nan
    class: n=35 median=0.910 p25=0.869 p75=0.954 #>0.5=35 #>0.7=35 #>0.9=19 #perfect=0


Ep 20/20:   0%|          | 0/131 [00:00<?, ?it/s]

[ep20] step 0/131 loss=0.1252
[ep20] step 50/131 loss=0.1378
[ep20] step 100/131 loss=0.1377


val:   0%|          | 0/7 [00:00<?, ?it/s]

=== Ep 20/20: loss=0.1370 val_ns22=0.9094 val_macro=0.9200 BEST lr_bk=0.00e+00 lr_hd=0.00e+00 (0.5min, total 10.8min) ===
    taxon: Aves=0.918 Amphibia=0.962 Insecta=1.000 Mammalia=nan Reptilia=nan
    class: n=35 median=0.914 p25=0.870 p75=0.953 #>0.5=35 #>0.7=35 #>0.9=19 #perfect=0

=== Stage 2A DONE. Best val_ns22=0.9094 ===


In [ ]:
# ============================================================
# Cell 14: Upload Stage 2A ckpt to Kaggle Dataset
# ============================================================
import json, shutil
UPLOAD_DIR = Path("/content/exp053_upload")
UPLOAD_DIR.mkdir(exist_ok=True, parents=True)

for src_file in [CFG.BEST_CKPT, CFG.LAST_CKPT, CFG.HIST_JSON]:
    if src_file.exists():
        shutil.copy(src_file, UPLOAD_DIR / src_file.name)
        print(f"  Copied: {src_file.name} ({src_file.stat().st_size/1e6:.1f} MB)")

USER = "maekeso"
SLUG = "birdclef2026-exp053-stage2a-effv2s"
meta = {
    "title": "BirdCLEF2026 exp053 Stage 2A EfficientNetV2-S",
    "id": f"{USER}/{SLUG}",
    "licenses": [{"name": "other"}],
}
(UPLOAD_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

import subprocess
def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env={**os.environ})
    print(r.stdout); print(r.stderr[:300] if r.stderr else "")
    return r.returncode

ret = run(f"kaggle datasets version -p {UPLOAD_DIR} -m 'Stage 2A best={best_val:.4f}' -r tar")
if ret != 0:
    print("Version up failed, creating new...")
    ret = run(f"kaggle datasets create -p {UPLOAD_DIR} -r tar")
if ret == 0:
    print(f"\nOK Dataset uploaded: https://www.kaggle.com/datasets/{USER}/{SLUG}")
else:
    print(f"\nUpload failed. Manual upload needed.")


  Copied: stage2a_best.pth (84.0 MB)
  Copied: stage2a_last.pth (84.0 MB)
  Copied: stage2a_history.json (0.0 MB)
Starting upload for file stage2a_last.pth
Upload successful: stage2a_last.pth (80MB)
Starting upload for file stage2a_history.json
Upload successful: stage2a_history.json (11KB)
Starting upload for file stage2a_best.pth
Upload successful: stage2a_best.pth (80MB)
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/CreateDatasetVersion


  0%|          | 0.00/80.1M [00:00<?, ?B/s]
  1%|          | 800k/80.1M [00:00<00:11, 7.14MB/s]
  2%|▏         | 1.47M/80.1M [00:00<00:31, 2.61MB/s]
  2%|▏         | 1.86M/80.1M [00:00<00:29, 2.74MB/s]
 12%|█▏        | 10.0M/80.1M [00:00<00:03, 21.7MB/s]
 19%|█▉        | 15.5M/80.1M [00:00<00:02, 2
Version up failed, creating new...
Starting upload for file stage2a_last.pth
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'
Upload successful: stage2

In [ ]:
# ============================================================
# Cell 15: Summary
# ============================================================
print(f"=== Stage 2A Summary ===")
print(f"  Stage 1 backbone: {STAGE1_CKPT}")
print(f"  Stage 1 val_ns22: {stage1_ckpt.get('val_ns22', 'n/a')}")
print(f"  Stage 2A epochs: {CFG.EPOCHS}")
print(f"  Best val_ns22: {best_val:.4f}")
print(f"  Total time: {(time.time() - start_t)/60:.1f} min")
print()
print(f"Best ckpt: {CFG.BEST_CKPT} ({CFG.BEST_CKPT.stat().st_size/1e6:.1f} MB)")
print()
print(f"Next: Inference NB → 4-way blend (exp048 + Stage 2A)")
print(f"  Expected blend LB: 0.951-0.955 (gold border 接近)")


=== Stage 2A Summary ===
  Stage 1 backbone: /content/drive/MyDrive/kaggle/birdclef2026/exp052/ckpt/stage1_backbone_best.pth
  Stage 1 val_ns22: 0.9423327938232607
  Stage 2A epochs: 20
  Best val_ns22: 0.9094
  Total time: 11.2 min

Best ckpt: /content/drive/MyDrive/kaggle/birdclef2026/exp053/ckpt/stage2a_best.pth (84.0 MB)

Next: Inference NB → 4-way blend (exp048 + Stage 2A)
  Expected blend LB: 0.951-0.955 (gold border 接近)


In [ ]:
# ============================================================
# Cell 16: Disconnect runtime
# ============================================================
print("Disconnecting in 30 sec...")
import time
time.sleep(30)
from google.colab import runtime
runtime.unassign()


Disconnecting in 30 sec...
